# Steel Production – Regression Model v2
This notebook replaces the RandomForestRegressor with **XGBoost** (Extreme Gradient Boosting),
a more powerful algorithm that builds trees sequentially, each correcting the errors of the previous one.

In [1]:
# ── Step 1: Import all required libraries ──────────────────────────────────────
import os                                    # for directory creation
import pandas as pd                          # for loading and handling table data
import numpy as np                           # for numerical operations
import matplotlib.pyplot as plt             # for plotting graphs
from xgboost import XGBRegressor             # XGBoost regression model
from sklearn.model_selection import train_test_split # splits data into train/test
from sklearn.metrics import mean_squared_error, r2_score  # measures model quality

In [2]:
# ── Step 2: Load the CSV file ───────────────────────────────────────────────────
# pd.read_csv() reads a CSV file and returns a DataFrame (a table-like structure)
DATA_PATH = "../../data/processed/normalized_train_data.csv"
df = pd.read_csv(DATA_PATH)

# Show the first 5 rows so we can inspect the data
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset loaded: 7642 rows, 22 columns


,output,input1,input2,input3,input4,input5,input6,input7,input8,input9,...,input12,input13,input14,input15,input16,input17,input18,input19,input20,input21
0,0.364444,0.0,0.570312,0.778443,0.543058,0.538462,0.621350,0.314136,0.634801,0.667774,...,0.566327,0.583333,0.361702,0.275362,0.107744,0.247059,0.063545,0.079330,0.496855,0.511770
1,0.408889,0.0,0.574219,0.784431,0.532513,0.538462,0.645285,0.418848,0.602578,0.634551,...,0.515306,0.277778,0.297872,0.275362,0.090909,0.200000,0.063545,0.081564,0.484277,0.450741
2,0.431111,0.0,0.589844,0.782934,0.534271,0.538462,0.645285,0.261780,0.624060,0.568106,...,0.553571,0.388889,0.297872,0.275362,0.090909,0.211765,0.063545,0.082682,0.471698,0.485615
3,0.440000,0.0,0.580078,0.782934,0.532513,0.538462,0.645285,0.261780,0.613319,0.601329,...,0.642857,0.388889,0.340426,0.275362,0.090909,0.258824,0.063545,0.083799,0.471698,0.572799
4,0.422222,0.0,0.589844,0.796407,0.525483,0.538462,0.645285,0.261780,0.613319,0.568106,...,0.655612,0.444444,0.340426,0.347826,0.090909,0.223529,0.063545,0.083799,0.471698,0.564080


In [3]:
# ── Step 3: Separate input features (X) and output target (y) ──────────────────
# The first column 'output' is what we want to predict (the target)
# All other columns are the input parameters the model learns from
y = df["output"]               # target column – the value we want to predict
X = df.drop("output", axis=1)  # all remaining columns – the input features

print(f"Input features (X): {X.shape[1]} columns")
print(f"Target values  (y): {y.shape[0]} rows")

Input features (X): 21 columns
Target values  (y): 7642 rows


In [ ]:
# ── Step 4: Define evaluation seeds ────────────────────────────────────────────
# Instead of a single train/test split, the model is trained and evaluated
# 10 times with different random seeds. Each run uses a different 80/20 split
# and a different model initialization, so mean ± std can be reported.
# The run with seed=42 is saved separately for the plots in Step 6.

SEEDS = [0, 1, 7, 13, 21, 42, 55, 77, 88, 99]

print(f"Evaluation will run over {len(SEEDS)} seeds: {SEEDS}")

In [ ]:
# ── Step 5: Train and evaluate XGBoost across all seeds ────────────────────────
# For each seed: split data, train a new model, predict on the test set,
# and record R² and RMSE. The run with seed=42 is saved as the reference
# model and predictions for the plots in the following steps.

r2_scores   = []
rmse_scores = []
model  = None   # reference model  (seed=42) used for feature importance
y_test = None   # reference test labels (seed=42) used for plotting
y_pred = None   # reference predictions (seed=42) used for plotting

for seed in SEEDS:
    X_train, X_test_s, y_train, y_test_s = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    m = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        verbosity=0
    )
    m.fit(X_train, y_train)
    y_p = m.predict(X_test_s)

    r2_scores.append(r2_score(y_test_s, y_p))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test_s, y_p)))

    # save the seed=42 run for plotting
    if seed == 42:
        model, y_test, y_pred = m, y_test_s, y_p

print("Training complete across all seeds.")

In [ ]:
# ── Step 6: Results – mean ± std across all runs ───────────────────────────────
r2_mean,   r2_std   = np.mean(r2_scores),   np.std(r2_scores)
rmse_mean, rmse_std = np.mean(rmse_scores), np.std(rmse_scores)

# single-run values from seed=42 (used in the plot title below)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Multi-run evaluation over {len(SEEDS)} seeds")
print(f"R²   : {r2_mean:.4f} +/- {r2_std:.4f}")
print(f"RMSE : {rmse_mean:.4f} +/- {rmse_std:.4f}")

In [ ]:
# ── Step 7: Plot – Actual vs. Predicted values ─────────────────────────────────
# Plots are based on the seed=42 reference run.
# The plot title shows mean ± std from all 10 runs.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left plot: Scatter plot of actual vs predicted ──────────────────────────
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.3, s=10, color="steelblue", label="Samples")

lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, color="red", linewidth=1.5, linestyle="--", label="Perfect fit")

ax.set_xlabel("Actual output (from CSV)")
ax.set_ylabel("Predicted output (model)")
ax.set_title(
    f"Actual vs. Predicted  [XGBoost, seed=42]\n"
    f"R² = {r2_mean:.4f} +/- {r2_std:.4f}  |  RMSE = {rmse_mean:.4f} +/- {rmse_std:.4f}"
)
ax.legend()
ax.grid(True, alpha=0.3)

# ── Right plot: Residuals ────────────────────────────────────────────────────
residuals = y_test.values - y_pred

ax2 = axes[1]
ax2.scatter(y_pred, residuals, alpha=0.3, s=10, color="darkorange")
ax2.axhline(0, color="red", linewidth=1.5, linestyle="--", label="Zero error")
ax2.set_xlabel("Predicted output")
ax2.set_ylabel("Residual (actual - predicted)")
ax2.set_title("Residual Plot  [XGBoost, seed=42]")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../../results/figures/actual_vs_predicted_v2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to results/figures/actual_vs_predicted_v2.png")